In [ ]:
import time, json
import os
import requests
import xml.etree.ElementTree as ET
from typing import List, Dict, Optional

from dotenv import load_dotenv
load_dotenv()
LAW_API=os.getenv("LAW_API")


BASE_URL = "https://www.law.go.kr/DRF/lawSearch.do"

def _get_text(elem: Optional[ET.Element]) -> str:
    if elem is None or elem.text is None:
        return ""
    return elem.text.strip()

def fetch_page(api_key: str, page: int, display: int = 100, session: Optional[requests.Session] = None) -> str:
    params = {
        "OC": api_key,
        "target": "law",
        "display": str(display),
        "page": str(page),
    }
    sess = session or requests.Session()
    resp = sess.get(BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    return resp.text

def parse_laws(xml_text: str) -> List[Dict[str, str]]:
    root = ET.fromstring(xml_text)
    results: List[Dict[str, str]] = []

    for law in root.findall(".//law"):
        name_ko = _get_text(law.find("법령명한글"))
        short_name = _get_text(law.find("법령약칭명"))
        law_id = _get_text(law.find("법령ID"))       # "001702"
        law_type = _get_text(law.find("법령구분명"))  # 법률/시행령/시행규칙 등

        if name_ko and law_id:
            results.append({
                "law_name": name_ko,
                "law_short_name": short_name,   # 없으면 ""로 저장됨
                "law_id": law_id,
                "law_type": law_type,
            })

    return results

def crawl_law_mapping(
    api_key: str,
    display: int = 100,
    start_page: int = 1,
    sleep_sec: float = 0.2,
    max_pages: Optional[int] = None,
    max_retries: int = 5,
) -> List[Dict[str, str]]:

    all_items: List[Dict[str, str]] = []
    sess = requests.Session()

    page = start_page
    while True:
        if max_pages is not None and (page - start_page) >= max_pages:
            break

        last_err = None
        for attempt in range(1, max_retries + 1):
            try:
                xml_text = fetch_page(api_key, page=page, display=display, session=sess)
                items = parse_laws(xml_text)
                break
            except Exception as e:
                last_err = e
                time.sleep(min(2 ** (attempt - 1), 10))
        else:
            raise RuntimeError(f"Failed page={page} after {max_retries} retries: {last_err}") from last_err

        if not items:
            print(f"[STOP] page={page}: no <law> entries.")
            break

        all_items.extend(items)
        print(f"[OK] page={page}: {len(items)} items (total={len(all_items)})")
        page += 1
        time.sleep(sleep_sec)

    return all_items


def save_json(items: List[Dict[str, str]], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False, indent=2)

def save_jsonl(items: List[Dict[str, str]], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for obj in items:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")


if __name__ == "__main__":
    API_KEY = LAW_API

    items = crawl_law_mapping(
        api_key=API_KEY,
        display=100,
        start_page=1,
        sleep_sec=0.5,
        max_pages=None,
    )

    # 전체(법률+시행령+시행규칙...) 저장
    # save_json(items, "law_map.json")
    save_jsonl(items, "../data/law_map.jsonl")

    # 법률만 따로 저장
    # only_law = [x for x in items if x.get("법령구분명") == "법률"]
    # save_json(only_law, "law_map_only_법률.json")
    # save_jsonl(only_law, "law_map_only_법률.jsonl")

    print(f"Done. total={len(items)}")


[OK] page=1: 100 items (total=100)
[OK] page=2: 100 items (total=200)
[OK] page=3: 100 items (total=300)
[OK] page=4: 100 items (total=400)
[OK] page=5: 100 items (total=500)
[OK] page=6: 100 items (total=600)
[OK] page=7: 100 items (total=700)
[OK] page=8: 100 items (total=800)
[OK] page=9: 100 items (total=900)
[OK] page=10: 100 items (total=1000)
[OK] page=11: 100 items (total=1100)
[OK] page=12: 100 items (total=1200)
[OK] page=13: 100 items (total=1300)
[OK] page=14: 100 items (total=1400)
[OK] page=15: 100 items (total=1500)
[OK] page=16: 100 items (total=1600)
[OK] page=17: 100 items (total=1700)
[OK] page=18: 100 items (total=1800)
[OK] page=19: 100 items (total=1900)
[OK] page=20: 100 items (total=2000)
[OK] page=21: 100 items (total=2100)
[OK] page=22: 100 items (total=2200)
[OK] page=23: 100 items (total=2300)
[OK] page=24: 100 items (total=2400)
[OK] page=25: 100 items (total=2500)
[OK] page=26: 100 items (total=2600)
[OK] page=27: 100 items (total=2700)
[OK] page=28: 100 i